<a href="https://colab.research.google.com/github/SattamAltwaim/KAUST-Mawhiba-IOAI-2026/blob/main/butterfly_competition/baseline_notebook.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>&nbsp;&nbsp;<a href="https://www.kaggle.com/t/41199ce08b5e47c896df8b6ac37760de" target="_blank"><img src="https://img.shields.io/badge/Kaggle-Join%20Competition-20BEFF?logo=kaggle&logoColor=white" alt="Join Competition"/></a>

# Butterfly Classification — Baseline

100 species of butterflies & moths. Pretrained **EfficientNet-B0**, minimal augmentation.

$$\text{Score} = \text{Macro F1} \times 100$$

In [ ]:
!pip install -q kagglehub

---
## Config

In [ ]:
IMG_SIZE       = 224
BATCH_SIZE     = 32
EPOCHS         = 5
LR             = 1e-3
NUM_WORKERS    = 2

IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

In [ ]:
import os
import kagglehub
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

---
## 1 — Data Loading

In [ ]:
# ════════════════════════════════════════════════════════════
#  DATA LOADING — do not modify
# ════════════════════════════════════════════════════════════

IMAGE_DATASET = "gpiosenka/butterfly-images40-species"
COMP_DATASET  = "sattamjaltwaim/butterfly-dataset"

img_root = kagglehub.dataset_download(IMAGE_DATASET)
csv_root = kagglehub.dataset_download(COMP_DATASET)

train_csv = pd.read_csv(os.path.join(csv_root, "train.csv"))
test_csv  = pd.read_csv(os.path.join(csv_root, "test.csv"))
label_map = pd.read_csv(os.path.join(csv_root, "label_map.csv"))

NUM_CLASSES = len(label_map)
idx_to_name = dict(zip(label_map["class_index"], label_map["class_name"]))

print(f"Train images : {len(train_csv):,}")
print(f"Test images  : {len(test_csv):,}")
print(f"Classes      : {NUM_CLASSES}")

---
## 2 — Quick EDA

In [ ]:
class_counts = train_csv["label"].value_counts().sort_index()
print(f"Images per class — min: {class_counts.min()}, max: {class_counts.max()}, "
      f"median: {class_counts.median():.0f}")

plt.figure(figsize=(14, 3))
plt.bar(class_counts.index, class_counts.values, width=1.0)
plt.xlabel("Class index")
plt.ylabel("Count")
plt.title("Training images per class")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
sample_rows = train_csv.sample(10, random_state=42)
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    img = Image.open(os.path.join(img_root, row["filepath"]))
    ax.imshow(img)
    ax.set_title(idx_to_name[row["label"]][:20], fontsize=8)
    ax.axis("off")
plt.suptitle("Sample training images — notice grayscale, background-heavy, and color variety")
plt.tight_layout()
plt.show()

---
## 3 — Dataset & DataLoader

In [ ]:
train_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ButterflyDataset(Dataset):
    def __init__(self, df, img_root, transform, has_label=True):
        self.df = df.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_root, row["filepath"]))
        img = img.convert("RGB")
        img = self.transform(img)
        if self.has_label:
            return img, int(row["label"])
        return img


train_ds = ButterflyDataset(train_csv, img_root, train_transforms, has_label=True)
test_ds  = ButterflyDataset(test_csv,  img_root, test_transforms,  has_label=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

---
## 4 — Model: Pretrained EfficientNet-B0

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)

model = model.to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable:  {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

---
## 5 — Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch + 1}/{EPOCHS}  loss={epoch_loss:.4f}  acc={epoch_acc:.3f}")

---
## 6 — Inference on Test Set

In [ ]:
model.eval()
all_preds = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

all_preds = np.array(all_preds)
print(f"Predictions: {len(all_preds)} images across {len(np.unique(all_preds))} unique classes")

---
## 7 — Generate Submission

In [ ]:
# ════════════════════════════════════════════════════════════
#  SUBMISSION GENERATOR — do not modify
# ════════════════════════════════════════════════════════════

def generate_submission(predictions, filename="submission.csv"):
    """Create a Kaggle-compatible submission CSV.

    Parameters
    ----------
    predictions : array-like of int
        Predicted class indices (0 .. NUM_CLASSES-1), one per test image.
    filename : str
        Output CSV path.
    """
    y = np.asarray(predictions, dtype=int)
    assert len(y) == len(test_csv), (
        f"Prediction length {len(y)} != test size {len(test_csv)}"
    )

    submission = pd.DataFrame({
        "id": test_csv["id"].values,
        "prediction": y.astype(float),
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename}  ({len(submission)} rows)")
    n_unique = len(np.unique(y))
    print(f"  Unique classes predicted: {n_unique}/{NUM_CLASSES}")


generate_submission(all_preds)